# Getting Started with Account Details Tool Development

Welcome to the Firewall Automation Hackathon! This notebook will guide you through building the Account Details tools.

## Overview

The Account Details tools provide AWS account metadata by querying DynamoDB. These tools will be integrated into the supervisor agent. The tools:
- Query AWS account metadata from DynamoDB
- Support fuzzy matching for account names (e.g., "prod" matches "RT-Prod")
- Look up accounts by ID, name, or CIDR range
- Provide account details including VPC CIDR ranges, environment, and owner information
- Use caching to improve performance

## Related Jira Task

- [FWAUTO-16: Account Details Agent implementation](https://your-jira-instance.atlassian.net/browse/FWAUTO-16)

## How to Get Started

Your task is to implement **DynamoDB query tools** that will be added to the supervisor agent in `agent/src/agent.py`.

### Authentication Setup

DynamoDB access requires cross-account role assumption:

```
IAM Role for Bedrock Agentcore ---cross-account---> DynamoDB
```

Make sure that the IAM role `arn:aws:iam::123456789012:role/YourAgentCoreExecutionRole` has the correct permissions to query DynamoDB table in the `PACE-Automation-PROD` account. Note down the permission changes and make sure to include the change in the `infra/cloudformation/app-template.yaml` file. 

### Examples
An example implementation using Strands is available in the `account_details_tool_example.py` file.

### Implementation Approaches

#### Option 1: DynamoDB as a Tool (Direct Implementation)
Implement DynamoDB query tools directly using boto3:
- Create tools for querying by name, ID, and CIDR
- Explore fuzzy matching with fuzzywuzzy library
- Add these tools to the supervisor agent

#### Option 2: DynamoDB MCP Server (Recommended to Explore)
After implementing Option 1, explore the AWS DynamoDB MCP Server for a simpler integration:
- https://awslabs.github.io/mcp/servers/dynamodb-mcp-server
- Provides standardized interface to DynamoDB
- Simplifies query operations
- Compare the implementation complexity with Option 1

### Key Libraries

- **boto3**: For DynamoDB access and STS AssumeRole
- **fuzzywuzzy**: For fuzzy string matching on account names
- **datetime**: For cache TTL management

### Reference Implementation

Study the `network_firewall_analyser_agent.py` in the firewall-logs-agent folder for:
- Cross-account role assumption pattern
- Tool definitions using `@tool` decorator
- Error handling and result formatting

## Step 1: Copy the Base Agent Code

Copy the supervisor agent code to your workspace so you can modify it.

In [ ]:
%%bash
# Copy the base agent code from the repository
cp -r /home/sagemaker-user/IST-AWS-Firewall-Automation/agent .

# Remove the existing bedrock_agentcore.yaml configuration
rm -f agent/.bedrock_agentcore.yaml

In [ ]:
# View the base agent python code
with open('agent/src/agent.py', 'r') as f:
    print(f.read())

## Step 2: Review the README

Study the README.md file in this folder to understand the DynamoDB schema, tool definitions, and implementation patterns.

In [ ]:
# View the README for implementation guidance
with open('README.md', 'r') as f:
    print(f.read())

## Step 3: Implement Account Details Tools

Replace the dummy `query_account_details` tool in `agent/src/agent.py` with real DynamoDB query functionality.

### Key Implementation Steps:

1. **Initialize DynamoDB client** - Create boto3 DynamoDB resource with assumed credentials
2. **Implement query tools** - Replace the dummy tool with real functionality:
   - Query by account name with fuzzy matching
   - Query by 12-digit account ID
   - Query by CIDR range
3. **Add fuzzy matching** - Use fuzzywuzzy library for partial name matching
4. **Add error handling** - Handle DynamoDB errors, missing accounts, role assumption failures

### Implementation Approach:

These are **tools for the supervisor agent** to query account metadata:
- Uses boto3 DynamoDB resource for queries
- Implements fuzzy matching for user-friendly account name lookup
- Caches results to reduce DynamoDB load (5-minute TTL)
- Provides structured account information
- The supervisor agent will use these tools when account details are needed

### Alternative: DynamoDB MCP Server

After implementing the direct DynamoDB integration (Option 1), explore the AWS DynamoDB MCP Server (Option 2):
- Simpler integration with standardized interface
- Compare implementation complexity and performance

In [ ]:
# View the current dummy implementation
with open('agent/src/agent.py', 'r') as f:
    content = f.read()
    # Find and display the query_account_details function
    start = content.find('def query_account_details')
    end = content.find('\n\n@tool', start)
    if start != -1:
        print(content[start:end if end != -1 else start+1000])

## Step 4: Test Your Implementation

Test the tool locally before deploying.

In [ ]:
# TODO: Add your test code here
# Example test:
# from agent.src.agent import query_account_details
# 
# # Test querying by name
# result = query_account_details(account_identifier="prod")
# print(f"Query by name result:\n{result}")
# 
# # Test querying by ID
# result = query_account_details(account_identifier="123456789012")
# print(f"Query by ID result:\n{result}")
# 
# # Test querying by CIDR
# result = query_account_details(account_identifier="10.10.0.0/16")
# print(f"Query by CIDR result:\n{result}")

## Step 5: Deploy to AWS

Deploy your updated supervisor agent (with the new account details tools) to AWS Bedrock AgentCore Runtime.

In [ ]:
import time
import boto3
from bedrock_agentcore_starter_toolkit import Runtime

# TODO: Set your agent name
agent_name = <AGENT_NAME>  # e.g., "firewall-supervisor-agent"

# Initialize the runtime toolkit
region = "ap-southeast-2"

agentcore_runtime = Runtime()

# Configure the deployment
response = agentcore_runtime.configure(
    agent_name=agent_name,
    entrypoint=<ENTRYPOINT>,  # TODO: Set your entrypoint file, e.g., "agent/src/agent.py"
    execution_role="arn:aws:iam::123456789012:role/YourAgentCoreExecutionRole",
    code_build_execution_role="arn:aws:iam::123456789012:role/YourCodeBuildRole",
    auto_create_ecr=True,
    requirements_file=<REQUIREMENTS_FILE>,  # TODO: Set your requirements file, e.g., "agent/src/requirements.txt"
    region=region,
    memory_mode="STM_ONLY",
)

print("Configuration completed:", response)

launch_result = agentcore_runtime.launch()
print("Launch completed:", launch_result.agent_arn)

# Wait for the agent to be ready
status_response = agentcore_runtime.status()
status = status_response.endpoint["status"]

end_status = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]
while status not in end_status:
    print(f"Waiting for deployment... Current status: {status}")
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]

if status == "READY":
    runtime_id = status_response.agent["agentRuntimeId"]

    # Update the runtime to be deployed in VPC
    client = boto3.client("bedrock-agentcore-control", region_name=region)

    response = client.update_agent_runtime(
        agentRuntimeId=runtime_id,
        networkConfiguration={
            "networkMode": "VPC",
            "networkModeConfig": {
                "subnets": ["subnet-xxxxxxxxxxxxxxxxx", "subnet-yyyyyyyyyyyyyyyyy"],
                "securityGroups": ["sg-xxxxxxxxxxxxxxxxx"],
            },
        },
        agentRuntimeArtifact={
            "containerConfiguration": {
                "containerUri": f"123456789012.dkr.ecr.ap-southeast-2.amazonaws.com/bedrock-agentcore-{agent_name}"
            }
        },
        roleArn="arn:aws:iam::123456789012:role/YourAgentCoreExecutionRole",
    )
    
print(f"Supervisor Agent deployed successfully with account details tools!")

## Step 6: Test End-to-End

Test the deployed agent through the Streamlit UI.

## Step 7: Push Your Changes

Create a branch and push your changes for review.

## Tips and Best Practices

### Security
- **Store DynamoDB table name and role ARN** in environment variables or AWS Secrets Manager
- **Use IAM roles** for authentication, not access keys
- **Implement proper error handling** to avoid leaking sensitive account information
- **Audit logging**: Log all account queries with user context

### DynamoDB Queries
- **Use caching**: Implement 5-minute TTL cache to reduce DynamoDB load
- **Batch queries**: Use `batch_get_item` when fetching multiple accounts
- **Handle missing data**: Return gracefully when account not found (don't raise errors)
- **Case insensitive**: Normalize strings to lowercase for matching

### Fuzzy Matching
- **Threshold**: Use score > 60 for acceptable matches
- **Multiple matches**: Return all matches sorted by relevance score
- **Did you mean?**: Suggest alternatives when no exact match found
- **Partial matching**: "prod" should match "RT-Prod", "RT-Prod-Web", etc.

### Performance
- **Lazy initialization**: Initialize DynamoDB client only when needed
- **Connection pooling**: Reuse boto3 resources across tool calls
- **Limit result sets**: Cap maximum results to prevent memory issues

### User Experience
- **Provide context**: Include confidence scores with fuzzy matches
- **Clear errors**: Explain what went wrong ("Account 'prodx' not found. Did you mean 'RT-Prod'?")
- **Rich responses**: Include all relevant metadata (CIDR, environment, owner)
- **Multiple matches**: Handle ambiguous queries gracefully

## Resources

- README: `README.md` in this folder
- Jira: [FWAUTO-16](https://your-jira-instance.atlassian.net/browse/FWAUTO-16)
- AWS DynamoDB MCP Server: https://awslabs.github.io/mcp/servers/dynamodb-mcp-server
- fuzzywuzzy Library: https://github.com/seatgeek/fuzzywuzzy
- boto3 DynamoDB Documentation: https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/dynamodb.html
- `./account_details_tool_example.py` for an old example implementation using Strands
- `/home/sagemaker-user/amazon-bedrock-agentcore-samples/` for more examples from AWS